# Pré-processamento

Os três conjuntos exportados na EDA são carregados e unidos em uma única tabela, e a coluna `conjunto` indica a origem de cada linha. A união é necessária porque as variáveis de janela (últimas 3, 6 e 9 semanas) precisam do histórico completo de cada produto em ordem cronológica: para prever a primeira semana de validação, por exemplo, é preciso enxergar as semanas finais do treino. Isso não gera vazamento, pois cada variável usa apenas semanas anteriores à prevista e os parâmetros aprendidos são calculados somente com o treino.

In [96]:
import pandas as pd
import numpy as np

tr = pd.read_csv('../data/raw/treino.csv', parse_dates=['semana'])
va = pd.read_csv('../data/raw/validacao.csv', parse_dates=['semana'])
te = pd.read_csv('../data/raw/teste.csv', parse_dates=['semana'])

tr['conjunto'], va['conjunto'], te['conjunto'] = 'treino', 'validacao', 'teste' #cria coluna nova pra identificar a origem

df = pd.concat([tr, va, te], ignore_index=True)

In [97]:
print(df.shape)                      # esperado: 3168 linhas (2119 + 523 + 526)
print(df['conjunto'].value_counts())

(3168, 20)
conjunto
treino       2119
teste         526
validacao     523
Name: count, dtype: int64


## Completando o painel produto × semana

In [98]:
semanas = sorted(df['no_semana'].unique())
primeira_semana = df.groupby('id_produto')['no_semana'].min() #guarda a primeira semana que cada produto aparece

#produto cartesiano produtos x semanas
produtos = pd.DataFrame({'id_produto': df['id_produto'].unique()})
semanas_df = pd.DataFrame({'no_semana': semanas})
painel = produtos.merge(semanas_df, how='cross')


#só a partir da primeira semana em que o produto apareceu
painel = painel[painel['no_semana'] >= painel['id_produto'].map(primeira_semana)]

#preenche com os dados reais, mantendo vazio onde não teve pedido
painel = painel.merge(df, on=['id_produto', 'no_semana'], how='left')

Como a tabela semanal só possui linhas quando houve pedido, as semanas sem demanda não estavam representadas, e o modelo enxergaria apenas semanas com venda. Para corrigir, foi criada uma linha para cada combinação de produto e semana, a partir da primeira semana em que o produto aparece. Assim, não são criados zeros antes de o produto existir.

In [99]:
#preenchendo com demanda zero as semanas sem pedido
painel['qtd_total'] = painel['qtd_total'].fillna(0)
painel['n_pedidos'] = painel['n_pedidos'].fillna(0)
painel['n_dias_com_pedido'] = painel['n_dias_com_pedido'].fillna(0)


#preenchendo curva e categoria
painel = painel.sort_values(['id_produto', 'no_semana'])
for col in ['curva', 'categoria']:
    painel[col] = painel.groupby('id_produto')[col].ffill()


#preenchendo a coluna conjunto
painel['conjunto'] = np.select(
    [painel['no_semana'] <= 9, painel['no_semana'] <= 11],
    ['treino', 'validacao'],
    default='teste'
)

print(painel['conjunto'].value_counts())
print(painel[['qtd_total', 'n_pedidos', 'n_dias_com_pedido']].isna().sum())  # deve ser 0
print(f"Linhas com demanda zero: {(painel['qtd_total'] == 0).sum()} ({(painel['qtd_total'] == 0).mean():.1%})")

conjunto
treino       2487
teste         680
validacao     670
Name: count, dtype: int64
qtd_total            0
n_pedidos            0
n_dias_com_pedido    0
dtype: int64
Linhas com demanda zero: 669 (17.4%)


As semanas sem pedido foram preenchidas com demanda zero (nas colunas `qtd_total`, `n_pedidos` e `n_dias_com_pedido`). As colunas `curva` e `categoria`, que são características fixas do produto, foram propagadas para as semanas novas. As métricas por pedido (`qtd_media`, `qtd_mediana`, `qtd_min` e `qtd_max`) permanecem vazias nessas semanas, pois não há pedidos a resumir. Por fim, a coluna `conjunto` foi preenchida a partir do número da semana.

Após o preenchimento, o conjunto de dados passou de 3.168 para 3.837 linhas. As 669 linhas adicionadas (17,4% do painel) correspondem a semanas sem pedidos, e nenhuma das colunas de demanda possui valores nulos.

## Tratamento das variáveis categóricas

Como visto na EDA, temos duas variáveis categóricas, `curva` e `categoria`, que são nulas para produtos que não aparecem na tabela de produtos. Como essa ausência é estrutural e não aleatória, os valores não foram imputados com estatísticas: foi criado o indicador `tem_info_produto`, que marca se o produto já possuía cadastro naquela semana, e os ausentes passaram a formar um nível próprio ("Sem cadastro").

As categorias com até 5 linhas no conjunto de treino foram agrupadas em um único nível ("Outras"), para evitar níveis com pouquíssima informação. A lista de categorias mantidas foi definida somente com o treino, de modo que categorias que só aparecem na validação ou no teste também são tratadas como "Outras".

In [100]:
#indicador se o produto existe na tabela de produtos
painel['tem_info_produto'] = painel['curva'].notna().astype(int)


# categorias raras: definidas só a partir do conjunto de treino
treino_painel = painel[painel['conjunto'] == 'treino']
contagem = treino_painel['categoria'].value_counts()
categorias_mantidas = contagem[contagem > 5].index   # mantém aquelas com mais de 5 linhas

sem_cadastro = painel['tem_info_produto'] == 0
painel['categoria'] = painel['categoria'].where(painel['categoria'].isin(categorias_mantidas), 'Outras')
painel.loc[sem_cadastro, 'categoria'] = 'Sem cadastro'

# curva: nulos viram um nível próprio
painel['curva'] = painel['curva'].fillna('Sem cadastro')

Após o tratamento, 32 das 36 categorias do treino foram mantidas, e 4 foram agrupadas em "Outras", o que corresponde a 38 linhas do painel (cerca de 1%). Cerca de 74% das linhas (2.828 de 3.837) estão no nível "Sem cadastro", proporção próxima à observada na EDA. A `curva` possui apenas os níveis A, B e "Sem cadastro".

A proporção de linhas com cadastro cresce ao longo do tempo (25,0% no treino, 28,2% na validação e 29,3% no teste), pois alguns produtos passam a ser cadastrados nas semanas finais.

In [101]:
print(f"Categorias mantidas: {len(categorias_mantidas)} de {contagem.shape[0]}\n")
print(painel['categoria'].value_counts().head(5), "\n")
print(painel['curva'].value_counts(), "\n")

print('Proporção de linhas com cadastro:')
print(painel.groupby('conjunto')['tem_info_produto'].mean().round(3), "\n")

print('Quantidade de linhas na categoria "Outras":', painel['categoria'].value_counts()['Outras'])

Categorias mantidas: 32 de 36

categoria
Sem cadastro    2828
Categoria 36      95
Categoria 28      95
Categoria 20      79
Categoria 10      47
Name: count, dtype: int64 

curva
Sem cadastro    2828
B                751
A                258
Name: count, dtype: int64 

Proporção de linhas com cadastro:
conjunto
teste        0.293
treino       0.250
validacao    0.282
Name: tem_info_produto, dtype: float64 

Quantidade de linhas na categoria "Outras": 38


## Tratamento das demais variáveis de produto

Como `ip_mediano`, `ip_medio`, `ip_min` e `ip_max` são altamente correlacionados (0,69 a 0,96 na EDA), apenas `ip_mediano` foi mantido. As colunas `n_dias_coleta` e `ips_nao_nulos` descrevem a coleta de dados e foram descartadas.

Os valores ausentes de `ip_mediano` foram preenchidos com o último valor conhecido do produto (`ffill`, que usa apenas semanas anteriores) e, quando ainda restava ausência, com a mediana do treino. O indicador `ip_ausente` marca as linhas em que o valor não foi observado, e uma cópia do valor original (`ip_observado`) foi mantida para a estimativa de elasticidade, que deve usar apenas valores reais. Em `pct_dias_sem_estoque`, a ausência foi preenchida com 0.

In [102]:
painel = painel.sort_values(['id_produto', 'no_semana'])

# IP original (sem preenchimento), para usar na elasticidade
painel['ip_observado'] = painel['ip_mediano']

# indicador: 1 se o IP não foi observado naquela semana
painel['ip_ausente'] = painel['ip_mediano'].isna().astype(int)

# mantém apenas o ip_mediano e descarta o que é redundante ou só da coleta
painel = painel.drop(columns=['ip_medio', 'ip_min', 'ip_max', 'n_dias_coleta', 'ips_nao_nulos'])

# último valor conhecido do produto (só olha para semanas anteriores)
painel['ip_mediano'] = painel.groupby('id_produto')['ip_mediano'].ffill()

# o que ainda estiver nulo recebe a mediana do treino
mediana_ip_treino = painel.loc[painel['conjunto'] == 'treino', 'ip_observado'].median()
painel['ip_mediano'] = painel['ip_mediano'].fillna(mediana_ip_treino)

# estoque: sem informação, assume 0 (a ausência é marcada por tem_info_produto)
painel['pct_dias_sem_estoque'] = painel['pct_dias_sem_estoque'].fillna(0)

In [103]:
print(painel[['ip_mediano', 'pct_dias_sem_estoque']].isna().sum())   # deve ser 0
print(f"Mediana do IP no treino: {mediana_ip_treino:.3f}")
print(painel.groupby('conjunto')['ip_ausente'].mean().round(3))

ip_mediano              0
pct_dias_sem_estoque    0
dtype: int64
Mediana do IP no treino: 1.060
conjunto
teste        0.821
treino       0.812
validacao    0.822
Name: ip_ausente, dtype: float64


## Criação de variáveis defasadas (lags) e janelas móveis

Para evitar vazamento de dados, as métricas observadas na própria semana t (como `qtd_media`, `qtd_max`, `n_pedidos` e `n_dias_com_pedido`) não podem ser usadas diretamente como features para prever `qtd_total` daquela mesma semana, pois derivam dos mesmos pedidos que compõem o alvo.

Para que o modelo enxergue apenas o histórico anterior, foi aplicado um deslocamento de 1 semana (`shift(1)`) agrupado por produto antes do cálculo das janelas:
* Janelas móveis (3, 6 e 9 semanas): Foram calculadas as médias móveis das últimas 3, 6 e 9 semanas (`_media_mov_3`, `_media_mov_6`, `_media_mov_9`) e o pico máximo das últimas 3 semanas (`_max_mov_3`), utilizando o parâmetro `min_periods=1` para garantir que produtos recém-lançados (que ainda não possuem o histórico completo de 3, 6 ou 9 semanas) não fiquem com valores vazios, permitindo que a média seja calculada com base nas semanas que já estão disponíveis em seu início de vida.

* Defasagens diretas (lags): Foram extraídos os valores exatos de 1, 2 e 3 semanas anteriores (`_lag_1`, `_lag_2` e `_lag_3`) para as variáveis de demanda, pedidos, preço (`ip_mediano`) e ruptura de estoque (`pct_dias_sem_estoque`).

Nas semanas sem movimento, as métricas condicionais por pedido (`qtd_media`, `qtd_mediana`, `qtd_min` e `qtd_max`) e os nulos residuais das primeiras semanas de histórico foram preenchidos com `0`, eliminando valores ausentes antes da etapa de modelagem.

In [104]:
# preenchendo as métricas por pedido com 0 nas semanas sem movimento
colunas_por_pedido = ['qtd_media', 'qtd_mediana', 'qtd_min', 'qtd_max']
for col in colunas_por_pedido:
    if col in painel.columns:
        painel[col] = painel[col].fillna(0)

painel = painel.sort_values(['id_produto', 'no_semana']) #garante a ordenação

# definindo quais colunas vão virar histórico (lags e janelas)
colunas_alvo = ['qtd_total', 'n_pedidos', 'n_dias_com_pedido', 'ip_mediano', 'pct_dias_sem_estoque'] + colunas_por_pedido

# criando as features de Janela Móvel (3, 6 e 9 semanas) usando shift(1)
for col in colunas_alvo:
    if col in painel.columns:
        # aplicando o shift(1) por produto para olhar estritamente para o passado
        serie_passada = painel.groupby('id_produto')[col].shift(1)
        
        # janela de 3 semanas (média e máximo)
        painel[f'{col}_media_mov_3'] = serie_passada.groupby(painel['id_produto']).rolling(window=3, min_periods=1).mean().reset_index(level=0, drop=True)
        painel[f'{col}_max_mov_3'] = serie_passada.groupby(painel['id_produto']).rolling(window=3, min_periods=1).max().reset_index(level=0, drop=True)
        
        # janela de 6 semanas (só média)
        painel[f'{col}_media_mov_6'] = serie_passada.groupby(painel['id_produto']).rolling(window=6, min_periods=1).mean().reset_index(level=0, drop=True)
        
        # janela de 9 semanas (só média)
        painel[f'{col}_media_mov_9'] = serie_passada.groupby(painel['id_produto']).rolling(window=9, min_periods=1).mean().reset_index(level=0, drop=True)
        
        # lags diretos (ex: o valor exato da semana passada - lag 1)
        painel[f'{col}_lag_1'] = serie_passada
        painel[f'{col}_lag_2'] = painel.groupby('id_produto')[col].shift(2)
        painel[f'{col}_lag_3'] = painel.groupby('id_produto')[col].shift(3)

Garantindo a integridade dos dados gerados:
* Preenchimento de nulos residuais: O uso de janelas móveis e defasagens pode gerar valores ausentes nas primeiras semanas de vida de um produto (onde ainda não há histórico completo) ou em métricas condicionais de semanas sem pedidos. Esses nulos foram preenchidos com `0`, assumindo que a ausência de histórico anterior ou de movimento equivale a zero.
* Inspeção: Uma visualização pontual por produto garante que o deslocamento (`shift`) operou corretamente, empurrando as informações para a frente sem misturar dados da semana atual com o passado, garantindo que não houve vazamento.

In [105]:
# identifica todas as colunas numéricas de histórico recém-criadas que possuem nulos
colunas_novas = [c for c in painel.columns if 'lag' in c or 'mov' in c]

# preenche os nulos restantes com 0 de todas as features recém-criadas
painel[colunas_novas] = painel[colunas_novas].fillna(0)

# confirma se ainda restou algum nulo nas colunas novas
print("Total de nulos restantes nas colunas de histórico:", painel[colunas_novas].isna().sum().sum())
print(f"Shape final do painel pronto para treino: {painel.shape}\n\n")

# conta o total de colunas recém-criadas para conferir os nomes
colunas_novas = [c for c in painel.columns if 'lag' in c or 'media' in c or 'max' in c]
print(f"Total de features de histórico criadas: {len(colunas_novas)}")

# escolhe um produto que tenha bastante histórico para inspecionar
exemplo_produto = painel['id_produto'].unique()[0]

# Ajustado para os nomes reais gerados no loop (_mov_)
colunas_para_olhar = ['id_produto', 'no_semana', 'qtd_total', 'qtd_total_lag_1', 'qtd_total_media_mov_3', 'qtd_total_max_mov_3']
print(painel[painel['id_produto'] == exemplo_produto][colunas_para_olhar].head(10))


Total de nulos restantes nas colunas de histórico: 0
Shape final do painel pronto para treino: (3837, 81)


Total de features de histórico criadas: 67
         id_produto  no_semana  qtd_total  qtd_total_lag_1  \
3533  1010954545613          4        2.0              0.0   
3534  1010954545613          5        0.0              2.0   
3535  1010954545613          6        0.0              0.0   
3536  1010954545613          7        0.0              0.0   
3537  1010954545613          8        1.0              0.0   
3538  1010954545613          9        0.0              1.0   
3539  1010954545613         10        0.0              0.0   
3540  1010954545613         11        0.0              0.0   
3541  1010954545613         12        0.0              0.0   
3542  1010954545613         13        0.0              0.0   

      qtd_total_media_mov_3  qtd_total_max_mov_3  
3533               0.000000                  0.0  
3534               2.000000                  2.0  
3535         

## Transformações e valores extremos

Para evitar que picos atípicos de vendas distorçam o aprendizado do algoritmo, aplicamos uma técnica de limitação de cauda (*clipping* / winsorização). O percentil 99 (`p99`) foi calculado estritamente sobre o conjunto de treino para evitar qualquer vazamento de dados do futuro ou da validação. O teto máximo definido foi propagado para todo o painel, rebaixando os valores mais extremos para o limite saudável de corte sem descartar nenhuma linha de histórico.

In [106]:
# calculando o P99 da quantidade total usando somente o conjunto de treino
p99_treino = painel[painel['conjunto'] == 'treino']['qtd_total'].quantile(0.99)
print(f"P99 calculado no treino para qtd_total: {p99_treino}")

# aplicando em todo o painel usando esse valor de referência
painel['qtd_total'] = painel['qtd_total'].clip(upper=p99_treino)

P99 calculado no treino para qtd_total: 1608.3999999999924


Para estabilizar a variância e comprimir a cauda longa da distribuição de vendas (comum em dados de varejo, onde muitos produtos vendem pouco e poucos vendem muito), aplicamos a transformação `log1p` ($\ln(1 + x)$). A operação foi aplicada tanto à variável alvo principal (`qtd_total`) quanto à métrica de volume auxiliar (`n_pedidos`).

In [107]:
# colunas de volume / quantidade que vão receber o log1p
colunas_log = ['qtd_total', 'n_pedidos']

for col in colunas_log:
    if col in painel.columns:
        painel[f'{col}_log'] = np.log1p(painel[col])

print(f"Colunas atuais no painel: {painel.shape[1]}")

Colunas atuais no painel: 83


Para garantir que o modelo não precise prever o comportamento de itens desconhecidos, filtramos a base de dados identificando todos os produtos presentes no conjunto de treino e removendo da validação e do teste aqueles itens órfãos que apareceriam sem histórico prévio. Essa limpeza evita distorções artificiais e garante que a avaliação do modelo reflita cenários reais de aprendizado.

In [108]:
# Identifica quais produtos estão presentes no conjunto de treino
produtos_treino = set(painel[painel['conjunto'] == 'treino']['id_produto'])

# Filtra o painel para manter na validação e no teste apenas os produtos que já existiam no treino
painel = painel[
    (painel['conjunto'] == 'treino') | 
    (painel['id_produto'].isin(produtos_treino))
]

print(f"Shape atualizado após remover esses produtos: {painel.shape}")

Shape atualizado após remover esses produtos: (3815, 83)


## Exportação dos dados

In [109]:
import os

output_dir = '../data/processed/elasticidade'
os.makedirs(output_dir, exist_ok=True)

conjuntos = ['treino', 'validacao', 'teste']

for nome in conjuntos:
    df_subset = painel[painel['conjunto'] == nome]
    caminho = os.path.join(output_dir, f'{nome}.csv')
    df_subset.to_csv(caminho, index=False)

In [110]:
painel

,id_produto,no_semana,semana,qtd_total,qtd_media,qtd_mediana,qtd_min,qtd_max,n_pedidos,n_dias_com_pedido,...,qtd_min_lag_3,qtd_max_media_mov_3,qtd_max_max_mov_3,qtd_max_media_mov_6,qtd_max_media_mov_9,qtd_max_lag_1,qtd_max_lag_2,qtd_max_lag_3,qtd_total_log,n_pedidos_log
3533,1010954545613,4,2026-01-19,2.0,1.000000,1.0,1.0,1.0,2.0,2.0,...,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,1.098612,1.098612
3534,1010954545613,5,NaT,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,1.000000,1.0,1.000000,1.000000,1.0,0.0,0.0,0.000000,0.000000
3535,1010954545613,6,NaT,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.500000,1.0,0.500000,0.500000,0.0,1.0,0.0,0.000000,0.000000
3536,1010954545613,7,NaT,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,1.0,0.333333,1.0,0.333333,0.333333,0.0,0.0,1.0,0.000000,0.000000
3537,1010954545613,8,2026-02-16,1.0,1.000000,1.0,1.0,1.0,1.0,1.0,...,0.0,0.000000,0.0,0.250000,0.250000,0.0,0.0,0.0,0.693147,0.693147
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3187,9986625254736,9,2026-02-23,10.0,2.000000,2.0,1.0,3.0,5.0,3.0,...,1.0,5.333333,10.0,5.000000,5.714286,3.0,3.0,10.0,2.397895,1.791759
3188,9986625254736,10,2026-03-02,43.0,1.592593,1.0,1.0,5.0,27.0,6.0,...,1.0,3.000000,3.0,4.666667,5.375000,3.0,3.0,3.0,3.784190,3.332205
3189,9986625254736,11,2026-03-09,41.0,1.464286,1.0,1.0,5.0,28.0,7.0,...,1.0,3.666667,5.0,4.833333,5.333333,5.0,3.0,3.0,3.737670,3.367296
3190,9986625254736,12,2026-03-16,31.0,1.291667,1.0,1.0,2.0,24.0,7.0,...,1.0,4.333333,5.0,4.833333,4.777778,5.0,5.0,3.0,3.465736,3.218876


## Preparação para a modelagem

Os arquivos `*_processado.csv` continuam como estão e alimentam o notebook de elasticidade. Daqui em diante, o `painel` é usado para gerar os conjuntos de treino, validação e teste que serão usados na modelagem.

Dada a limitação que nos foi imposta de que cerca de 75% dos produtos estão sem cadastro (uma % maior sem o índice de preço), optamos por aplicar dois métodos distintos de modelagem e compará-los:

* `sem_faltantes/`: remove os registros em que o índice de preço (IP) não foi observado, sem imputar nenhum valor de IP.
* `imputado/`: mantém todos os registros e usa o IP imputado (último valor conhecido do produto e, na falta dele, a mediana do treino), com os indicadores `tem_info_produto` e `ip_ausente`.

As duas versões passam pelas mesmas etapas de codificação das categóricas e padronização, sempre aprendendo os parâmetros apenas com o treino.

### Variáveis explicativas

Entram como variáveis explicativas os lags e as janelas móveis (que usam só semanas anteriores à prevista), o `ip_mediano` da semana, os indicadores `tem_info_produto` e `ip_ausente`, e a `curva` e a `categoria` codificadas. Ficam de fora as colunas observadas na própria semana (`qtd_media`, `qtd_mediana`, `qtd_min`, `qtd_max`, `n_pedidos`, `n_dias_com_pedido` e `pct_dias_sem_estoque`), porque vêm dos mesmos pedidos que formam o alvo, e a `ip_observado`, que serve apenas à elasticidade. O alvo é `qtd_total_log`.

In [111]:
sufixos_hist = ('_lag_1', '_lag_2', '_lag_3', '_media_mov_3', '_media_mov_6', '_media_mov_9', '_max_mov_3')
cols_hist = [c for c in painel.columns if c.endswith(sufixos_hist)]

# históricos de volume (quantidades e contagens), que também recebem log1p
cols_volume = [c for c in cols_hist if c.startswith(('qtd_', 'n_pedidos', 'n_dias_com_pedido'))]

cols_continuas = cols_hist + ['ip_mediano']
cols_binarias = ['tem_info_produto', 'ip_ausente']
alvo = 'qtd_total_log'

print(f"Variáveis contínuas: {len(cols_continuas)} (das quais {len(cols_volume)} de volume) | binárias: {len(cols_binarias)}")

Variáveis contínuas: 64 (das quais 49 de volume) | binárias: 2


### Registros com índice de preço faltante

A tabela abaixo mostra por que o IP falta no treino. É a partir dela que as duas versões são definidas.

In [112]:
treino_p = painel[painel['conjunto'] == 'treino']

situacao = np.select(
    [
        treino_p['tem_info_produto'] == 0,
        treino_p['ip_observado'].notna(),
        treino_p['qtd_total'] == 0,
    ],
    ['Sem cadastro', 'Com IP observado', 'Com cadastro, semana sem pedido'],
    default='Com cadastro e pedido, mas IP ausente',
)
pd.Series(situacao).value_counts().to_frame('linhas').assign(pct=lambda t: (t['linhas'] / t['linhas'].sum() * 100).round(1))

,linhas,pct
Sem cadastro,1866,75.0
Com IP observado,467,18.8
"Com cadastro, semana sem pedido",80,3.2
"Com cadastro e pedido, mas IP ausente",74,3.0


O IP está ausente em cerca de 81% das linhas de treino. A maior parte (75%) é de produtos sem cadastro, que nunca têm IP. As demais são produtos cadastrados em semanas sem IP, entre elas as semanas sem pedido, que não têm IP porque a tabela semanal de produtos foi unida a partir da tabela de pedidos.

* Na versão `sem_faltantes`, todas essas linhas são removidas e ficam somente as 467 linhas com IP observado. Como as semanas sem pedido têm IP faltante, elas também saem, e essa versão não contém nenhuma semana com demanda zero.
* Na versão `imputado`, todas as linhas são mantidas, com o IP já preenchido no tratamento feito acima.

### Codificação de variáveis categóricas e padronização

* **Variáveis categóricas**: `curva` e `categoria` recebem one-hot encoding, conforme a análise feita na EDA. Os níveis são os do treino, e qualquer categoria que não exista no treino da versão entra em "Outras". Colunas sem variação no treino (como `curva_Sem_cadastro` na versão `sem_faltantes`) são removidas.
* **Variáveis de volume**: recebem `log1p`, como decidido na EDA para as variáveis de quantidade, o que reduz a assimetria dos lags e das janelas.
* **Padronização z-score**: `(x - média) / desvio`, com média e desvio calculados somente no treino e aplicados à validação e ao teste. Indicadores 0/1, colunas do one-hot e o alvo não são padronizados.

In [113]:
from sklearn.preprocessing import StandardScaler

def one_hot(serie, niveis, prefixo):
    serie = serie.where(serie.isin(niveis))   # nível desconhecido fica com todas as colunas em 0
    return pd.get_dummies(serie.astype(pd.CategoricalDtype(niveis)), prefix=prefixo, dtype=int)

def preparar_dataset(entrada, remover_faltantes):
    d = entrada.copy()
    if remover_faltantes:
        d = d[d['ip_observado'].notna()].copy()

    # semanas sem pedido não têm data no painel; ela é recuperada pelo número da semana
    d['semana'] = d['no_semana'].map(entrada.dropna(subset=['semana']).groupby('no_semana')['semana'].first())

    d[cols_volume] = np.log1p(d[cols_volume])
    idx_treino = d.index[d['conjunto'] == 'treino']

    # one-hot com os níveis do treino
    niveis_curva = sorted(d.loc[idx_treino, 'curva'].unique())
    niveis_cat = sorted(set(d.loc[idx_treino, 'categoria'].unique()) | {'Outras'})
    d['categoria'] = d['categoria'].where(d['categoria'].isin(niveis_cat), 'Outras')
    dummies = pd.concat([one_hot(d['curva'], niveis_curva, 'curva'), one_hot(d['categoria'], niveis_cat, 'categoria')], axis=1)
    dummies.columns = [c.replace('categoria_Categoria ', 'categoria_').replace(' ', '_') for c in dummies.columns]

    # remove o que não varia no treino
    dummies = dummies.loc[:, dummies.loc[idx_treino].nunique() > 1]
    binarias = [c for c in cols_binarias if d.loc[idx_treino, c].nunique() > 1]

    # z-score com média e desvio do treino
    d[cols_continuas] = StandardScaler().fit(d.loc[idx_treino, cols_continuas]).transform(d[cols_continuas])

    final = pd.concat([d[['id_produto', 'no_semana', 'semana', 'conjunto', 'qtd_total', alvo] + cols_continuas + binarias], dummies], axis=1)
    return {c: final[final['conjunto'] == c].drop(columns='conjunto').reset_index(drop=True) for c in ['treino', 'validacao', 'teste']}


In [114]:
datasets = {
    'sem_faltantes': preparar_dataset(painel, remover_faltantes=True),
    'imputado': preparar_dataset(painel, remover_faltantes=False),
}

resumo = [
    {'dataset': nome, 'conjunto': conj, 'linhas': len(df_c), 'produtos': df_c['id_produto'].nunique(),
     'colunas': df_c.shape[1], 'pct_demanda_zero': round((df_c['qtd_total'] == 0).mean() * 100, 1)}
    for nome, conjuntos in datasets.items() for conj, df_c in conjuntos.items()
]
pd.DataFrame(resumo)

,dataset,conjunto,linhas,produtos,colunas,pct_demanda_zero
0,sem_faltantes,treino,467,80,104,0.0
1,sem_faltantes,validacao,119,69,104,0.0
2,sem_faltantes,teste,121,68,104,0.0
3,imputado,treino,2487,332,108,14.8
4,imputado,validacao,664,332,108,22.0
5,imputado,teste,664,332,108,22.9


### Exportação dos dados

Armazenamos as versões em caminhos distintos (`imputado` e `sem_faltantes`), com os arquivos `treino.csv`, `validacao.csv` e `teste.csv`.

In [115]:
import os

for nome, conjuntos in datasets.items():
    pasta = f'../data/processed/{nome}'
    os.makedirs(pasta, exist_ok=True)
    for conj, df_c in conjuntos.items():
        df_c.to_csv(f'{pasta}/{conj}.csv', index=False)
    print(f"{pasta}: " + ', '.join(f"{conj}.csv ({len(df_c)} linhas)" for conj, df_c in conjuntos.items()))

../data/processed/sem_faltantes: treino.csv (467 linhas), validacao.csv (119 linhas), teste.csv (121 linhas)
../data/processed/imputado: treino.csv (2487 linhas), validacao.csv (664 linhas), teste.csv (664 linhas)
